# RAG Evaluation: Amazon Product Dataset 2020

Covers task backlog §1.5, evaluating [1.4 hybrid retrieval.ipynb](1.4%20hybrid%20retrieval.ipynb)'s
`search()` against a small labeled eval set:

- 14 queries across both collections, with budget/brand filters on a few
- precision@5 and recall@5 per query, and averaged
- failure cases inspected, not just scored

**Ground truth methodology (a real limitation, stated up front):** there's no human-labeled
relevance data for this dataset. "Relevant" here is defined by a **keyword/substring proxy**:
does the product title contain the query's key term(s)? This is a reasonable, cheap
approximation, not gold-standard labels: it will under-count true positives that use a
synonym instead of the literal keyword, and over-count near-misses that happen to share a
word. Treat the numbers below as directional, not authoritative.

No new dependencies. Reuses `search()` from §1.4.

In [1]:
# !pip install pandas sentence-transformers chromadb python-dotenv

## Setup: load data, model, collections, and `search()` from §1.4

In [2]:
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(REPO_ROOT, ".env"))

INPUT_CSV = os.path.join(REPO_ROOT, "Code", "1. Data Pipeline & Indexing", "processed", "products_clean.csv")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
CHROMA_PERSIST_DIR_HK = os.path.join(REPO_ROOT, os.environ.get(
    "CHROMA_PERSIST_DIR_HOME_KITCHEN", "Code/1. Data Pipeline & Indexing/processed/chroma_db_home_kitchen"
))
CHROMA_PERSIST_DIR_ALL = os.path.join(REPO_ROOT, os.environ.get(
    "CHROMA_PERSIST_DIR_ALL", "Code/1. Data Pipeline & Indexing/processed/chroma_db_all"
))
CHROMA_COLLECTION_HK = os.environ.get("CHROMA_COLLECTION", "products_home_kitchen")
CHROMA_COLLECTION_ALL = os.environ.get("CHROMA_COLLECTION_ALL", "products_all")

import chromadb
from sentence_transformers import SentenceTransformer

df = pd.read_csv(INPUT_CSV)
model = SentenceTransformer(EMBEDDING_MODEL)
client_hk = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR_HK)
client_all = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR_ALL)
collection_hk = client_hk.get_collection(CHROMA_COLLECTION_HK)
collection_all = client_all.get_collection(CHROMA_COLLECTION_ALL)
print(f"{CHROMA_COLLECTION_HK}: {collection_hk.count():,}  |  {CHROMA_COLLECTION_ALL}: {collection_all.count():,}")


def search(query, collection=None, top_k=5, budget_min=None, budget_max=None,
           category=None, brand=None, overfetch=30):
    collection = collection or collection_hk
    where_clauses = []
    if budget_min is not None:
        where_clauses.append({"price": {"$gte": budget_min}})
    if budget_max is not None:
        where_clauses.append({"price": {"$lte": budget_max}})
    if category is not None:
        where_clauses.append({"category_top": category})
    where = None
    if len(where_clauses) == 1:
        where = where_clauses[0]
    elif len(where_clauses) > 1:
        where = {"$and": where_clauses}

    query_embedding = model.encode([query])[0].tolist()
    n_results = overfetch if brand else top_k
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results, where=where)

    candidates = []
    for meta, dist, doc in zip(results["metadatas"][0], results["distances"][0], results["documents"][0]):
        if brand and brand.lower() not in meta.get("brand", "").lower():
            continue
        candidates.append({"doc_id": meta["id"], "title": meta.get("title", doc),
                            "price": meta.get("price"), "brand": meta.get("brand"),
                            "distance": dist})
    return candidates[:top_k]

products_home_kitchen: 706  |  products_all: 9,930


## Build the eval set

Each entry: a natural-language query, which collection it targets, any filters, and a
keyword used to compute the ground-truth relevant set (substring match on `title`,
case-insensitive).

In [3]:
hk_df = df[df["category_top"] == "Home & Kitchen"]

EVAL_SET = [
    {"query": "insulated lunch box for kids", "collection": collection_hk, "scope_df": hk_df, "keyword": "lunch box"},
    {"query": "kids bedding set", "collection": collection_hk, "scope_df": hk_df, "keyword": "bedding"},
    {"query": "toy storage organizer", "collection": collection_hk, "scope_df": hk_df, "keyword": "storage"},
    {"query": "birthday party decorations", "collection": collection_hk, "scope_df": hk_df, "keyword": "party"},
    {"query": "kids table and chairs", "collection": collection_hk, "scope_df": hk_df, "keyword": "table"},
    {"query": "paper plates", "collection": collection_hk, "scope_df": hk_df, "keyword": "plates"},
    {"query": "area rug for kids room", "collection": collection_hk, "scope_df": hk_df, "keyword": "rug"},
    {"query": "duvet cover set", "collection": collection_hk, "scope_df": hk_df, "keyword": "duvet"},
    {"query": "party balloons", "collection": collection_hk, "scope_df": hk_df, "keyword": "balloon"},
    {"query": "step stool", "collection": collection_hk, "scope_df": hk_df, "keyword": "stool"},
    {"query": "lunch box under $15", "collection": collection_hk, "scope_df": hk_df, "keyword": "lunch box",
     "filters": {"budget_max": 15}},
    {"query": "insulated lunch box", "collection": collection_hk, "scope_df": hk_df, "keyword": "lunch box",
     "filters": {"brand": "Wildkin"}},
    {"query": "dinosaur toy", "collection": collection_all, "scope_df": df, "keyword": "dinosaur"},
    {"query": "board game for family night", "collection": collection_all, "scope_df": df, "keyword": "board game"},
]

for e in EVAL_SET:
    ids = e["scope_df"][e["scope_df"]["title"].str.contains(e["keyword"], case=False, na=False)]["id"]
    e["relevant_ids"] = set(ids)
    if "filters" in e and "budget_max" in e["filters"]:
        priced = e["scope_df"][e["scope_df"]["id"].isin(e["relevant_ids"]) & (e["scope_df"]["price"] <= e["filters"]["budget_max"])]
        e["relevant_ids"] = set(priced["id"])
    if "filters" in e and "brand" in e["filters"]:
        b = e["filters"]["brand"].lower()
        matched = e["scope_df"][e["scope_df"]["id"].isin(e["relevant_ids"]) & e["scope_df"]["brand"].fillna("").str.lower().str.contains(b)]
        e["relevant_ids"] = set(matched["id"])

print(f"{len(EVAL_SET)} queries. Relevant-set sizes:")
for e in EVAL_SET:
    print(f"  {e['query']!r}: {len(e['relevant_ids'])} relevant (by {e['keyword']!r})")

14 queries. Relevant-set sizes:
  'insulated lunch box for kids': 9 relevant (by 'lunch box')
  'kids bedding set': 58 relevant (by 'bedding')
  'toy storage organizer': 22 relevant (by 'storage')
  'birthday party decorations': 61 relevant (by 'party')
  'kids table and chairs': 41 relevant (by 'table')
  'paper plates': 6 relevant (by 'plates')
  'area rug for kids room': 33 relevant (by 'rug')
  'duvet cover set': 12 relevant (by 'duvet')
  'party balloons': 13 relevant (by 'balloon')
  'step stool': 24 relevant (by 'stool')
  'lunch box under $15': 8 relevant (by 'lunch box')
  'insulated lunch box': 4 relevant (by 'lunch box')
  'dinosaur toy': 44 relevant (by 'dinosaur')
  'board game for family night': 94 relevant (by 'board game')


## Run precision@5 / recall@5

**Precision@5** = of the 5 results returned, how many were relevant (quality of the top 5). **Recall@5** = of all relevant items in the dataset, how many made it into those 5 (coverage).

Real `search()` calls against the real index, not simulated.

In [4]:
K = 5
rows = []
for e in EVAL_SET:
    filters = e.get("filters", {})
    results = search(e["query"], collection=e["collection"], top_k=K, **filters)
    retrieved_ids = {r["doc_id"] for r in results}
    hits = retrieved_ids & e["relevant_ids"]

    precision = len(hits) / K
    recall = len(hits) / len(e["relevant_ids"]) if e["relevant_ids"] else float("nan")

    rows.append({
        "query": e["query"], "precision@5": precision, "recall@5": recall,
        "hits": len(hits), "relevant_total": len(e["relevant_ids"]),
        "results": results,
    })

results_df = pd.DataFrame(rows)[["query", "precision@5", "recall@5", "hits", "relevant_total"]]
print(results_df.to_string(index=False))
print()
print(f"Mean precision@5: {results_df['precision@5'].mean():.2f}")
print(f"Mean recall@5:    {results_df['recall@5'].mean():.2f}")

                       query  precision@5  recall@5  hits  relevant_total
insulated lunch box for kids          1.0  0.555556     5               9
            kids bedding set          1.0  0.086207     5              58
       toy storage organizer          1.0  0.227273     5              22
  birthday party decorations          0.8  0.065574     4              61
       kids table and chairs          1.0  0.121951     5              41
                paper plates          0.8  0.666667     4               6
      area rug for kids room          1.0  0.151515     5              33
             duvet cover set          0.8  0.333333     4              12
              party balloons          1.0  0.384615     5              13
                  step stool          1.0  0.208333     5              24
         lunch box under $15          1.0  0.625000     5               8
         insulated lunch box          0.8  1.000000     4               4
                dinosaur toy          

## Failure cases

Look at the actual worst-scoring queries, not just their numbers: is it a real retrieval
miss, or a ground-truth-proxy artifact?

In [5]:
worst = results_df.sort_values("precision@5").head(3)
for _, row in worst.iterrows():
    entry = next(e for e in EVAL_SET if e["query"] == row["query"])
    print(f"=== {row['query']!r}  (precision@5={row['precision@5']:.2f}, recall@5={row['recall@5']:.2f}) ===")
    print(f"Ground truth ({row['relevant_total']} by keyword {entry['keyword']!r}):")
    gt_titles = entry["scope_df"][entry["scope_df"]["id"].isin(entry["relevant_ids"])]["title"].head(5)
    for t in gt_titles:
        print(f"    - {t[:70]}")
    print("Retrieved:")
    match = next(r for r in rows if r["query"] == row["query"])
    for r in match["results"]:
        tag = "✓" if r["doc_id"] in entry["relevant_ids"] else " "
        print(f"  [{tag}] [{r['distance']:.3f}] {r['title'][:65]}")
    print()

=== 'board game for family night'  (precision@5=0.60, recall@5=0.03) ===
Ground truth (94 by keyword 'board game'):
    - GreenBrier Games Yashima Legends from Fairytale Board Game
    - Ninja Division NAS Howl & Yip Board Game
    - Star Trek Frontiers (Star Trek Themed Mage Knight) Board Game
    - The Meeples Board Game Addict T-Shirt
    - Celtic Dice Bag Board Game
Retrieved:
  [✓] [0.750] Fireside Games Grackles - Board Games for Families - Board Games 
  [✓] [0.781] Brain Games Voila! Board Game - A Fast and Fun Family Board Game 
  [✓] [0.819] Hey! Play! 7-in-1 Classic Wooden Board Game Set – Old Fashioned F
  [ ] [0.858] Family Feud Strikeout Card Game
  [ ] [0.868] Family Feud Trivia Box Card Game

=== 'birthday party decorations'  (precision@5=0.80, recall@5=0.07) ===
Ground truth (61 by keyword 'party'):
    - Beistle 50354 Party Supplies, 8" x 51⁄2" x 51⁄2", Multicolored
    - AMERICAN GREETINGS Rainbow Latex Balloons| Pack of 20 | Party Decor
    - #40 Celebration Candle 

## Conclusion

Retrieval quality is solid: precision@5 averaged 0.90 across 14 test queries. Full write-up: [rag_eval_plan.md](../../Project%20Overview/rag_eval_plan.md).